# A full business solution

## Now we will take our project from Day 1 to the next level

### BUSINESS CHALLENGE:

Create a product that builds a Brochure for a company to be used for prospective clients, investors and potential recruits.

We will be provided a company name and their primary website.

See the end of this notebook for examples of real-world business applications.

And remember: I'm always available if you have problems or ideas! Please do reach out.

In [ ]:
# imports
# If these fail, please check you're running from an 'activated' environment with (llms) in the command prompt

import os
import json
from dotenv import load_dotenv
from IPython.display import Markdown, display, update_display
from scraper import fetch_website_links, fetch_website_contents
from openai import OpenAI

In [ ]:
# Initialize and constants

load_dotenv(override=True)
api_key = os.getenv('OPENAI_API_KEY')

if api_key and api_key.startswith('sk-proj-') and len(api_key)>10:
    print("API key looks good so far")
else:
    print("There might be a problem with your API key? Please visit the troubleshooting notebook!")
    
MODEL = 'gpt-5-nano'
openai = OpenAI()

In [13]:

from openai import OpenAI

OLLAMA_BASE_URL = "http://localhost:11434/v1"

MODEL = 'llama3.2'
# MODEL = 'deepseek-r1:14b'
ollama = OpenAI(base_url=OLLAMA_BASE_URL, api_key='ollama')
# api key in local = can be anything

In [3]:
links = fetch_website_links("https://edwarddonner.com")
links

['https://edwarddonner.com/',
 'https://edwarddonner.com/curriculum/',
 'https://edwarddonner.com/proficient/',
 'https://edwarddonner.com/connect-four/',
 'https://edwarddonner.com/outsmart/',
 'https://edwarddonner.com/about-me-and-about-nebula/',
 'https://edwarddonner.com/posts/',
 'https://edwarddonner.com/',
 'https://news.ycombinator.com',
 'https://nebula.io/?utm_source=ed&utm_medium=referral',
 'https://www.prnewswire.com/news-releases/wynden-stark-group-acquires-nyc-venture-backed-tech-startup-untapt-301269512.html',
 'https://edwarddonner.com/curriculum/',
 'https://edwarddonner.com/2026/02/17/ai-coder-vibe-coder-to-agentic-engineer/',
 'https://edwarddonner.com/2026/02/17/ai-coder-vibe-coder-to-agentic-engineer/',
 'https://edwarddonner.com/2026/01/04/ai-builder-with-n8n-create-agents-and-voice-agents/',
 'https://edwarddonner.com/2026/01/04/ai-builder-with-n8n-create-agents-and-voice-agents/',
 'https://edwarddonner.com/2025/11/11/ai-live-event/',
 'https://edwarddonner.co

## First step: Have GPT-5-nano figure out which links are relevant

### Use a call to gpt-5-nano to read the links on a webpage, and respond in structured JSON.  
It should decide which links are relevant, and replace relative links such as "/about" with "https://company.com/about".  
We will use "one shot prompting" in which we provide an example of how it should respond in the prompt.

This is an excellent use case for an LLM, because it requires nuanced understanding. Imagine trying to code this without LLMs by parsing and analyzing the webpage - it would be very hard!

Sidenote: there is a more advanced technique called "Structured Outputs" in which we require the model to respond according to a spec. We cover this technique in Week 8 during our autonomous Agentic AI project.

In [4]:
link_system_prompt = """
You are provided with a list of links found on a webpage.
You are able to decide which of the links would be most relevant to include in a brochure about the company,
such as links to an About page, or a Company page, or Careers/Jobs pages.
You should respond in JSON as in this example:

{
    "links": [
        {"type": "about page", "url": "https://full.url/goes/here/about"},
        {"type": "careers page", "url": "https://another.full.url/careers"}
    ]
}
"""

In [5]:
def get_links_user_prompt(url):
    user_prompt = f"""
Here is the list of links on the website {url} -
Please decide which of these are relevant web links for a brochure about the company, 
respond with the full https URL in JSON format.
Do not include Terms of Service, Privacy, email links.

Links (some might be relative links):

"""
    links = fetch_website_links(url)
    user_prompt += "\n".join(links)
    return user_prompt

In [6]:
print(get_links_user_prompt("https://edwarddonner.com"))


Here is the list of links on the website https://edwarddonner.com -
Please decide which of these are relevant web links for a brochure about the company, 
respond with the full https URL in JSON format.
Do not include Terms of Service, Privacy, email links.

Links (some might be relative links):

https://edwarddonner.com/
https://edwarddonner.com/curriculum/
https://edwarddonner.com/proficient/
https://edwarddonner.com/connect-four/
https://edwarddonner.com/outsmart/
https://edwarddonner.com/about-me-and-about-nebula/
https://edwarddonner.com/posts/
https://edwarddonner.com/
https://news.ycombinator.com
https://nebula.io/?utm_source=ed&utm_medium=referral
https://www.prnewswire.com/news-releases/wynden-stark-group-acquires-nyc-venture-backed-tech-startup-untapt-301269512.html
https://edwarddonner.com/curriculum/
https://edwarddonner.com/2026/02/17/ai-coder-vibe-coder-to-agentic-engineer/
https://edwarddonner.com/2026/02/17/ai-coder-vibe-coder-to-agentic-engineer/
https://edwarddonner.

In [ ]:
def select_relevant_links(url):
    response = openai.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": link_system_prompt},
            {"role": "user", "content": get_links_user_prompt(url)}
        ],
        response_format={"type": "json_object"}
    )
    result = response.choices[0].message.content
    links = json.loads(result)
    return links
    

In [15]:
def select_relevant_links(url):
    response = ollama.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": link_system_prompt},
            {"role": "user", "content": get_links_user_prompt(url)}
        ],
        response_format={"type": "json_object"}
    )
    result = response.choices[0].message.content
    links = json.loads(result)
    return links
    

In [16]:
select_relevant_links("https://edwarddonner.com")

{'links': [{'type': 'home page', 'url': 'https://edwarddonner.com/'},
  {'type': 'about me and about nebula page',
   'url': 'https://edwarddonner.com/about-me-and-about-nebula/'},
  {'type': 'curriculum page', 'url': 'https://edwarddonner.com/curriculum/'},
  {'type': 'proficient page', 'url': 'https://edwarddonner.com/proficient/'},
  {'type': 'connect four game page',
   'url': 'https://edwarddonner.com/connect-four/'},
  {'type': 'outsmart tool page', 'url': 'https://edwarddonner.com/outsmart/'},
  {'type': 'LinkedIn profile',
   'url': 'https://www.linkedin.com/in/eddonner/'}]}

In [ ]:
def select_relevant_links(url):
    print(f"Selecting relevant links for {url} by calling {MODEL}")
    response = openai.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": link_system_prompt},
            {"role": "user", "content": get_links_user_prompt(url)}
        ],
        response_format={"type": "json_object"}
    )
    result = response.choices[0].message.content
    links = json.loads(result)
    print(f"Found {len(links['links'])} relevant links")
    return links

In [20]:
def select_relevant_links(url):
    print(f"Selecting relevant links for {url} by calling {MODEL}")
    response = ollama.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": link_system_prompt},
            {"role": "user", "content": get_links_user_prompt(url)}
        ],
        response_format={"type": "json_object"}
    )
    result = response.choices[0].message.content
    links = json.loads(result)
    print(f"Found {len(links['links'])} relevant links")
    return links

In [ ]:
select_relevant_links("https://edwarddonner.com")

In [21]:
select_relevant_links("https://huggingface.co")

Selecting relevant links for https://huggingface.co by calling llama3.2
Found 9 relevant links


{'links': [{'type': 'About page', 'url': 'https://huggingface.co'},
  {'type': 'Company page',
   'url': 'https://huggingface.co/datasets/huggingface/documentation-images/resolve/main/blog/chinese-language-blog/wechat.jpg'},
  {'type': 'Blog', 'url': 'https://blog.huggingface.co'},
  {'type': 'GitHub', 'url': 'https://github.com/huggingface'},
  {'type': 'Twitter', 'url': 'https://twitter.com/huggingface'},
  {'type': 'LinkedIn', 'url': 'https://www.linkedin.com/company/huggingface/'},
  {'type': 'Discord Join page', 'url': 'https://join.discord.huggingface.co'},
  {'type': 'Chat', 'url': 'https://chat.huggingface.co'},
  {'type': 'Documentation Hub', 'url': 'https://huggingface.co/docs'}]}

## Second step: make the brochure!

Assemble all the details into another prompt to GPT-5-nano

In [50]:
def fetch_page_and_all_relevant_links(url):
    contents = fetch_website_contents(url)
    relevant_links = select_relevant_links(url)
    result = f"## Landing Page:\n\n{contents}\n## Relevant Links:\n"
    print(relevant_links['links'])
    for link in relevant_links['links']:
        result += f"\n\n### Link: {link['type']}\n"
        print(link["url"])
        # if link["url"] in ["https://blog.huggingface.co", "https://brand.huggingface.co"]:
        #     continue
        try:
            result += fetch_website_contents(link["url"])
        except:
            pass
    return result

In [29]:
print(fetch_page_and_all_relevant_links("https://huggingface.co"))

Selecting relevant links for https://huggingface.co by calling llama3.2
Found 5 relevant links
[{'type': 'about page', 'url': 'https://huggingface.co'}, {'type': 'home page', 'url': 'https://huggingface.co'}, {'type': 'team page', 'url': 'https://status.huggingface.co/'}, {'type': 'blog link', 'url': 'https://discuss.huggingface.co'}, {'type': 'join/join discord', 'url': 'https://apply.workable.com/huggingface/'}]
https://huggingface.co
https://huggingface.co
https://status.huggingface.co/
https://discuss.huggingface.co
https://apply.workable.com/huggingface/
## Landing Page:

Hugging Face – The AI community building the future.

Hugging Face
Models
Datasets
Spaces
Community
Docs
Enterprise
Pricing
Log In
Sign Up
The AI community building the future.
The platform where the machine learning community collaborates on models, datasets, and applications.
Explore AI Apps
or
Browse 2M+ models
Trending on
this week
Models
Qwen/Qwen3.5-35B-A3B
Updated
1 day ago
•
378k
•
663
Qwen/Qwen3.5-27B
Up

In [60]:
# brochure_system_prompt = """
# You are an assistant that analyzes the contents of several relevant pages from a company website
# and creates a short brochure about the company for prospective customers, investors and recruits.
# Respond in markdown without code blocks.
# Include details of company culture, customers and careers/jobs if you have the information.
# """

# Or uncomment the lines below for a more humorous brochure - this demonstrates how easy it is to incorporate 'tone':

brochure_system_prompt = """
You are an assistant that analyzes the contents of several relevant pages from a company website
and creates a short, humorous, entertaining, witty brochure about the company for prospective customers, investors and recruits.
Respond in markdown without code blocks.
Include details of company culture, customers and careers/jobs if you have the information.
"""


In [31]:
def get_brochure_user_prompt(company_name, url):
    user_prompt = f"""
You are looking at a company called: {company_name}
Here are the contents of its landing page and other relevant pages;
use this information to build a short brochure of the company in markdown without code blocks.\n\n
"""
    user_prompt += fetch_page_and_all_relevant_links(url)
    user_prompt = user_prompt[:5_000] # Truncate if more than 5,000 characters
    return user_prompt

In [34]:
get_brochure_user_prompt("HuggingFace", "https://huggingface.co")

Selecting relevant links for https://huggingface.co by calling llama3.2
Found 7 relevant links
[{'type': 'Company page', 'url': 'https://huggingface.co'}, {'type': 'About our organization', 'url': 'https://www.linkedin.com/company/huggingface/'}, {'type': 'Blog', 'url': 'https://doc.rust-lang.org/rustrc/2018/v0.11.0-changelog.html'}, {'type': 'GitHub account', 'url': 'https://github.com/huggingface'}, {'type': 'Twitter profile', 'url': 'https://twitter.com/huggingface'}, {'type': 'Discord server', 'url': 'https://discord.com/invite/huggingface'}, {'type': 'About team', 'url': 'https://apply.workable.com/huggingface/'}]
https://huggingface.co
https://www.linkedin.com/company/huggingface/
https://doc.rust-lang.org/rustrc/2018/v0.11.0-changelog.html
https://github.com/huggingface
https://twitter.com/huggingface
https://discord.com/invite/huggingface
https://apply.workable.com/huggingface/


'\nYou are looking at a company called: HuggingFace\nHere are the contents of its landing page and other relevant pages;\nuse this information to build a short brochure of the company in markdown without code blocks.\n\n\n## Landing Page:\n\nHugging Face – The AI community building the future.\n\nHugging Face\nModels\nDatasets\nSpaces\nCommunity\nDocs\nEnterprise\nPricing\nLog In\nSign Up\nThe AI community building the future.\nThe platform where the machine learning community collaborates on models, datasets, and applications.\nExplore AI Apps\nor\nBrowse 2M+ models\nTrending on\nthis week\nModels\nQwen/Qwen3.5-35B-A3B\nUpdated\n1 day ago\n•\n378k\n•\n663\nQwen/Qwen3.5-27B\nUpdated\n3 days ago\n•\n172k\n•\n429\nunsloth/Qwen3.5-35B-A3B-GGUF\nUpdated\nabout 19 hours ago\n•\n350k\n•\n347\nQwen/Qwen3.5-122B-A10B\nUpdated\n4 days ago\n•\n120k\n•\n341\nQwen/Qwen3.5-397B-A17B\nUpdated\n5 days ago\n•\n889k\n•\n1.12k\nBrowse 2M+ models\nSpaces\nRunning\non\nZero\nFeatured\n1.74k\nQwen Image Mu

In [ ]:
def create_brochure(company_name, url):
    response = openai.chat.completions.create(
        model="gpt-4.1-mini",
        messages=[
            {"role": "system", "content": brochure_system_prompt},
            {"role": "user", "content": get_brochure_user_prompt(company_name, url)}
        ],
    )
    result = response.choices[0].message.content
    display(Markdown(result))

In [61]:
def create_brochure(company_name, url):
    response = ollama.chat.completions.create(
        model="llama3.2",
        messages=[
            {"role": "system", "content": brochure_system_prompt},
            {"role": "user", "content": get_brochure_user_prompt(company_name, url)}
        ],
    )
    result = response.choices[0].message.content
    display(Markdown(result))

In [53]:
create_brochure("HuggingFace", "https://huggingface.co")

Selecting relevant links for https://huggingface.co by calling llama3.2
Found 7 relevant links
[{'type': 'About company', 'url': 'https://huggingface.co/'}, {'type': 'Careers/Jobs', 'url': 'https://apply.workable.com/huggingface/'}, {'type': 'Inference models', 'url': 'inference/models'}, {'type': 'Models', 'url': 'https://huggingface.co/models/'}, {'type': 'Datasets', 'url': 'spaces/datasets/peteromallet/dataclaw-peteromallet'}, {'type': 'Pricing endpoints', 'url': '/pricing#endpoints'}, {'type': 'Enterprise', 'url': '/enterprise'}]
https://huggingface.co/
https://apply.workable.com/huggingface/
inference/models
https://huggingface.co/models/
spaces/datasets/peteromallet/dataclaw-peteromallet
/pricing#endpoints
/enterprise


### Hugging Face Brochure

**Mission & Purpose**

Hugging Face is the AI community building the future. Our platform provides a collaboration space for machine learning enthusiasts to create, discover, and work together on models, datasets, and applications.

**What We Do**

* Host and collaborate on unlimited public models, datasets, and applications
* Offer open-source tools and libraries to accelerate ML development
* Support various modalities such as text, image, video, audio, and 3D

**Features & Applications**

* Browse over 2M+ pre-trained models and explore their capabilities
* Build your portfolio by sharing your work with the world and building an ML profile
* Leverage our advanced AI models for tasks like:
	+ Text generation
	+ Image-to-text translation
	+ Video generation
	+ Speech synthesis

**Community & Collaboration**

Our community is diverse and talented, with individuals from all over the world contributing to machine learning projects. Join us to:

* Share your expertise and learn from others
* Collaborate on projects that push AI boundaries
* Stay up-to-date with industry trends and announcements

### Why Partner with Hugging Face

By partnering with us, you'll gain access to cutting-edge AI tools and expertise, enabling you to accelerate your ML development and innovation.

**Join the Hugging Face Community**

Ready to be part of the AI revolution? Sign up for our platform today and start collaborating on machine learning projects that shape the future!

## Finally - a minor improvement

With a small adjustment, we can change this so that the results stream back from OpenAI,
with the familiar typewriter animation

In [ ]:
def stream_brochure(company_name, url):
    stream = openai.chat.completions.create(
        model="gpt-4.1-mini",
        messages=[
            {"role": "system", "content": brochure_system_prompt},
            {"role": "user", "content": get_brochure_user_prompt(company_name, url)}
          ],
        stream=True
    )    
    response = ""
    display_handle = display(Markdown(""), display_id=True)
    for chunk in stream:
        response += chunk.choices[0].delta.content or ''
        update_display(Markdown(response), display_id=display_handle.display_id)

In [62]:
def stream_brochure(company_name, url):
    stream = ollama.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": brochure_system_prompt},
            {"role": "user", "content": get_brochure_user_prompt(company_name, url)}
          ],
        stream=True
    )    
    response = ""
    display_handle = display(Markdown(""), display_id=True)
    for chunk in stream:
        response += chunk.choices[0].delta.content or ''
        update_display(Markdown(response), display_id=display_handle.display_id)

In [59]:
stream_brochure("HuggingFace", "https://huggingface.co")

Selecting relevant links for https://huggingface.co by calling llama3.2
Found 7 relevant links
[{'type': 'Home', 'url': 'https://huggingface.co'}, {'type': 'Models', 'url': 'https://huggingface.co/models'}, {'type': 'Datasets', 'url': 'https://huggingface.co/datasets'}, {'type': 'Spaces', 'url': 'https://huggingface.co/spaces'}, {'type': 'Documentation', 'url': 'https://huggingface.co/docs'}, {'type': 'Company Page', 'url': 'https://huggingface.com/'}, {'type': 'About Us/Changelog', 'url': 'https://huggingface.com/changelog'}]
https://huggingface.co
https://huggingface.co/models
https://huggingface.co/datasets
https://huggingface.co/spaces
https://huggingface.co/docs
https://huggingface.com/
https://huggingface.com/changelog


# Hugging Face: Empowering the Future of Artificial Intelligence

Hugging Face is a leading platform for machine learning communities to collaborate, create, and innovate. Our mission is to build an open-source ecosystem that empowers developers to push the boundaries of artificial intelligence.

## Mission and Values

At Hugging Face, we believe in the power of collaboration and community-driven innovation. Our values reflect our passion for:

* **Open-source**: We believe that open-source technology should be accessible to everyone. That's why we provide a large collection of pre-trained models and datasets for anyone to use.
* **Innovation**: We're committed to developing cutting-edge AI technologies that solve real-world problems.
* **Community**: Our platform is designed to bring together machine learning practitioners, researchers, and educators from around the world.

## What We Offer

Hugging Face provides a comprehensive suite of tools and resources for building, training, and deploying machine learning models. Our key offerings include:

* **Models**: A vast library of pre-trained models, including text, image, video, audio, and 3D models.
* **Datasets**: Access to over 500,000+ datasets, carefully curated to support AI development.
* **Spaces**: An unlimited collaboration platform for hosting public models, datasets, and applications.
* **Documentation**: Comprehensive guides and tutorials to help you get started with our tools.

## Success Stories

Our community has created an array of innovative applications using our platform. From text-to-text translation services to image generation software, the possibilities are endless. Some notable examples include:

* **LLaMA**: A revolutionary model for human-to-human conversation
* **Jan**: An AI-powered drawing tool that lets users create art with ease

## Careers and Opportunities

We're a diverse team of AI enthusiasts dedicated to shaping the future of machine learning. If you share our passion, we invite you to join us:

* **Join our community**: Connect with like-minded individuals who are pushing the boundaries of AI.
* **Become a contributor**: Contribute your expertise, ideas, and code to help shape the future of Hugging Face.
* **Pursue a career in AI**: We offer opportunities for machine learning engineers, researchers, and educators.

## How You Can Help Us Shape the Future

If you're passionate about empowering the next generation of AI thinkers join us:

### Resources
- https://huggingface.co/home
  - Get access to a vast collection pre-trained AI models.
  - Discover datasets for your machine learning projects.
    - Browse 2M+ models and access a massive range of datasets, including text-to-text models images 
      generated from text input. This is available all year round.

- https://huggingface.co/spaces


Your role on our team would play in:
1) Collaboration and communication with other partners to ensure maximum knowledge transfer to meet requirements for all users worldwide
2) Developing software applications as per user needs and market trends.
3) Helping trainees develop skills through continuous learning programs.

**Conclusion**

At Hugging Face, we're committed to building an inclusive community that empowers the development of cutting-edge AI technologies. If you share our passion for AI innovation, join us in shaping the future of machine learning today.

In [63]:
# Try changing the system prompt to the humorous version when you make the Brochure for Hugging Face:

stream_brochure("HuggingFace", "https://huggingface.co")

Selecting relevant links for https://huggingface.co by calling llama3.2
Found 4 relevant links
[{'type': 'About/Aliases page', 'url': 'https://huggingface.co alliance'}, {'type': 'Company/Brand page', 'url': 'https://huggingface.co/brand'}, {'type': 'Blog page', 'url': 'https://blog.huggingface.co'}, {'type': 'GitHub page', 'url': 'https://github.com/huggingface'}]
https://huggingface.co alliance
https://huggingface.co/brand
https://blog.huggingface.co
https://github.com/huggingface


### Hugging Face: Revolutionizing AI for a Brighter Future

#### Meet the Pioneers of AI Collaboration

At Hugging Face, we're passionate about empowering the next generation of machine learning engineers, scientists, and end users to build an open and ethical AI future together. Our collaboration platform provides the tools and resources needed to explore, discover, and experiment with open-source ML.

#### Unleash Your Potential

*   **Access 2M+ Models**: From image classification to natural language processing, our models cover a wide range of applications.
*   **Discover 500k+ Datasets**: Expand your knowledge and improve your models with high-quality datasets.
*   **Join 1M+ Applications**: Share your work, collaborate with others, and build your portfolio.

#### Community-Driven Innovation

Our community is at the heart of our innovation. With over 2 million registered users, we're a hive of activity where you can share, discover, and experiment with AI.

#### ExploreAI Apps and Spaces

*   **Browse AI Apps**: From image-to-video conversion to text-to-speech, explore our extensive collection of applications.
*   **Join Spaces**: Run your own models, collaborate with others, and accelerate your progress.

#### Our Story

Founded by Colin Riepl in 2016, Hugging Face has grown into a leading platform for machine learning collaboration. With a talented science team and a fast-growing community, we're shaping the future of AI together.

### Join the Revolution

Visit our website to learn more about our platform, explore our resources, and join the Hugging Face community today!

#### [Learn More](HuggingFace Website)

#### [Join Our Community](HuggingFace Sign Up)

<table style="margin: 0; text-align: left;">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/business.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#181;">Business applications</h2>
            <span style="color:#181;">In this exercise we extended the Day 1 code to make multiple LLM calls, and generate a document.

This is perhaps the first example of Agentic AI design patterns, as we combined multiple calls to LLMs. This will feature more in Week 2, and then we will return to Agentic AI in a big way in Week 8 when we build a fully autonomous Agent solution.

Generating content in this way is one of the very most common Use Cases. As with summarization, this can be applied to any business vertical. Write marketing content, generate a product tutorial from a spec, create personalized email content, and so much more. Explore how you can apply content generation to your business, and try making yourself a proof-of-concept prototype. See what other students have done in the community-contributions folder -- so many valuable projects -- it's wild!</span>
        </td>
    </tr>
</table>

<table style="margin: 0; text-align: left;">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/important.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#900;">Before you move to Week 2 (which is tons of fun)</h2>
            <span style="color:#900;">Please see the week1 EXERCISE notebook for your challenge for the end of week 1. This will give you some essential practice working with Frontier APIs, and prepare you well for Week 2.</span>
        </td>
    </tr>
</table>

<table style="margin: 0; text-align: left;">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/resources.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#f71;">A reminder on 3 useful resources</h2>
            <span style="color:#f71;">1. The resources for the course are available <a href="https://edwarddonner.com/2024/11/13/llm-engineering-resources/">here.</a><br/>
            2. I'm on LinkedIn <a href="https://www.linkedin.com/in/eddonner/">here</a> and I love connecting with people taking the course!<br/>
            3. I'm trying out X/Twitter and I'm at <a href="https://x.com/edwarddonner">@edwarddonner<a> and hoping people will teach me how it's done..  
            </span>
        </td>
    </tr>
</table>

<table style="margin: 0; text-align: left;">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/thankyou.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#090;">Finally! I have a special request for you</h2>
            <span style="color:#090;">
                My editor tells me that it makes a MASSIVE difference when students rate this course on Udemy - it's one of the main ways that Udemy decides whether to show it to others. If you're able to take a minute to rate this, I'd be so very grateful! And regardless - always please reach out to me at ed@edwarddonner.com if I can help at any point.
            </span>
        </td>
    </tr>
</table>